In [16]:
import pandas as pd # main data tool (like Excel in Python)
import numpy as np # math & number operations
import matplotlib.pyplot as plt # basic charts
import seaborn as sns # nicer looking charts
import warnings
warnings.filterwarnings('ignore') # hide unnecessary warning messages
pd.set_option('display.max_columns', None) # show ALL columns
print('All libraries imported successfully!')

All libraries imported successfully!


In [17]:
df = pd.read_csv('D:\Technocolabs\Delivery Five Cities Datasets\delivery_delivery_jl.csv') # reads the file
print('File loaded!')
print('Number of rows :', df.shape[0]) # how many records
print('Number of columns :', df.shape[1]) # how many field

File loaded!
Number of rows : 31415
Number of columns : 17


In [3]:
df.head()


,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds
0,3322376,31,Jilin,4849,126.56526,43.84112,94,14,09-25 08:08:00,09-25 08:08:00,126.56614,43.87092,09-25 11:32:00,09-25 11:32:00,126.56919,43.84248,925
1,4093119,31,Jilin,4849,126.56519,43.84110,94,14,08-21 09:11:00,08-21 09:11:00,126.56611,43.87081,08-21 15:00:00,08-21 15:00:00,126.56939,43.84269,821
2,36226,31,Jilin,4849,126.56987,43.85017,235,14,06-08 15:42:00,06-08 15:42:00,126.56612,43.87074,06-08 17:24:00,06-08 17:24:00,126.57628,43.84771,608
3,3950697,31,Jilin,4849,126.56984,43.85005,235,14,09-03 14:03:00,09-03 14:03:00,NaN,NaN,09-03 16:31:00,09-03 16:31:00,126.56815,43.85131,903
4,4455630,31,Jilin,4849,126.56991,43.85006,235,14,06-07 14:54:00,06-07 14:54:00,126.56597,43.87104,06-07 16:58:00,06-07 16:58:00,126.57030,43.84985,607


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31415 entries, 0 to 31414
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           31415 non-null  int64  
 1   region_id          31415 non-null  int64  
 2   city               31415 non-null  object 
 3   courier_id         31415 non-null  int64  
 4   lng                31415 non-null  float64
 5   lat                31415 non-null  float64
 6   aoi_id             31415 non-null  int64  
 7   aoi_type           31415 non-null  int64  
 8   accept_time        31415 non-null  object 
 9   accept_gps_time    31415 non-null  object 
 10  accept_gps_lng     30460 non-null  float64
 11  accept_gps_lat     30460 non-null  float64
 12  delivery_time      31415 non-null  object 
 13  delivery_gps_time  31415 non-null  object 
 14  delivery_gps_lng   31415 non-null  float64
 15  delivery_gps_lat   31415 non-null  float64
 16  ds                 314

In [7]:
df.isnull().sum()

order_id               0
region_id              0
city                   0
courier_id             0
lng                    0
lat                    0
aoi_id                 0
aoi_type               0
accept_time            0
accept_gps_time        0
accept_gps_lng       955
accept_gps_lat       955
delivery_time          0
delivery_gps_time      0
delivery_gps_lng       0
delivery_gps_lat       0
ds                     0
dtype: int64

In [18]:
before = len(df)
df = df.dropna(subset=['accept_gps_lng', 'accept_gps_lat'])
df = df.reset_index(drop=True)
after = len(df)

print(f'Rows before : {before:,}')
print(f'Rows after  : {after:,}')
print(f'Rows removed: {before - after:,}')

Rows before : 31,415
Rows after  : 30,460
Rows removed: 955


In [19]:
df.isnull().sum().sum() == 0, 'Unexpected missing values found!'
print('Confirmed: No missing values in delivery_cq dataset.')

Confirmed: No missing values in delivery_cq dataset.


In [20]:
df['gps_synced_accept'] = df['accept_time'] == df['accept_gps_time']
df['gps_synced_delivery'] = df['delivery_time'] == df['delivery_gps_time']

In [21]:
print('GPS synced at accept :', df['gps_synced_accept'].sum())
print('GPS synced at delivery :', df['gps_synced_delivery'].sum())

GPS synced at accept : 30460
GPS synced at delivery : 30460


In [22]:
print(type(df['accept_time'][0])) # will show <class 'str'> = text
print(df['accept_time'][0]) # shows: 10-22 10:26:00

<class 'str'>
09-25 08:08:00


In [23]:
time_cols = ['accept_time', 'accept_gps_time',
'delivery_time', 'delivery_gps_time']
for col in time_cols:
    df[col] = pd.to_datetime('2023-' + df[col],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce') # bad values → NaT


In [24]:
print('After conversion:')
print(df[time_cols].dtypes)

After conversion:
accept_time          datetime64[ns]
accept_gps_time      datetime64[ns]
delivery_time        datetime64[ns]
delivery_gps_time    datetime64[ns]
dtype: object


In [25]:
# Example: How many minutes from accept to delivery?
df['accept_to_delivery_min'] = (
(df['delivery_time'] - df['accept_time']).dt.total_seconds() / 60)

In [26]:
print(df['accept_to_delivery_min'].describe().round(1))

count    30460.0
mean       201.9
std        141.1
min          0.0
25%         98.0
50%        174.0
75%        274.0
max       3573.0
Name: accept_to_delivery_min, dtype: float64


In [27]:
duplicates = df.duplicated(subset=['order_id'])
print('Number of duplicate rows:', duplicates.sum())

Number of duplicate rows: 0


In [28]:
before = len(df)
df = df.drop_duplicates(subset=['order_id'], keep='first')
df = df.reset_index(drop=True) # re-number the rows from 0
after = len(df)
print(f'Rows before: {before:,}')
print(f'Rows after : {after:,}')
print(f'Duplicates removed: {before - after:,}')

Rows before: 30,460
Rows after : 30,460
Duplicates removed: 0


In [29]:
col = 'accept_to_delivery_min'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = (df[col] < lower) | (df[col] > upper)
print(f'Outlier rows found: {outliers.sum():,}')
print(f'Valid range: {lower:.0f} to {upper:.0f} minutes')

Outlier rows found: 864
Valid range: -166 to 538 minutes


In [30]:
df = df[df['accept_to_delivery_min'] >= 0].reset_index(drop=True)
print('Rows after removing negatives:', len(df))

Rows after removing negatives: 30460


In [31]:
df['accept_to_delivery_min'] = df['accept_to_delivery_min'].clip(upper=upper)
print('Max duration after clipping:', df['accept_to_delivery_min'].max())


Max duration after clipping: 538.0


In [32]:
# Cell 10a — Extract hour, day, month from accept_time
df['accept_hour'] = df['accept_time'].dt.hour # 0 to 23
df['accept_day'] = df['accept_time'].dt.day # 1 to 31
df['accept_month'] = df['accept_time'].dt.month # 1 to 12
print(df[['accept_time','accept_hour','accept_day','accept_month']].head(3))
# Cell 10b — Group hour into time-of-day buckets
def time_of_day(hour):
    if 0 <= hour < 6: return 'Night'
    elif 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else: return 'Night'
df['time_of_day'] = df['accept_hour'].apply(time_of_day)
print(df['time_of_day'].value_counts())

          accept_time  accept_hour  accept_day  accept_month
0 2023-09-25 08:08:00            8          25             9
1 2023-08-21 09:11:00            9          21             8
2 2023-06-08 15:42:00           15           8             6
time_of_day
Morning      25442
Afternoon     4581
Evening        435
Night            2
Name: count, dtype: int64


In [33]:
df['gps_drift'] = ((
(df['delivery_gps_lng'] - df['accept_gps_lng'])**2 +
(df['delivery_gps_lat'] - df['accept_gps_lat'])**2
) ** 0.5)
print(df['gps_drift'].describe().round(4))

count    30460.0000
mean         0.0690
std          2.4264
min          0.0000
25%          0.0139
50%          0.0243
75%          0.0351
max        133.9792
Name: gps_drift, dtype: float64


In [34]:
df['delivery_speed_proxy'] = df['gps_drift'] / (df['accept_to_delivery_min'] + 1)
print('Speed proxy stats:')
print(df['delivery_speed_proxy'].describe().round(4))

Speed proxy stats:
count    30460.0000
mean         0.0005
std          0.0224
min          0.0000
25%          0.0001
50%          0.0001
75%          0.0002
max          2.7909
Name: delivery_speed_proxy, dtype: float64


In [35]:
df['delivery_month'] = df['delivery_time'].dt.month
print(df['delivery_month'].value_counts().sort_index())

delivery_month
5     1963
6     5224
7     6576
8     8326
9     7309
10    1062
Name: count, dtype: int64


In [36]:
print('Final dataset shape:', df.shape)
print()
print('Missing values left:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print()
print('New columns created:')
new_cols = ['gps_synced_accept', 'gps_synced_delivery',
'accept_to_delivery_min', 'accept_hour', 'accept_day',
'accept_month', 'time_of_day', 'gps_drift',
'delivery_speed_proxy', 'delivery_month']
print(new_cols)

Final dataset shape: (30460, 27)

Missing values left:
Series([], dtype: int64)

New columns created:
['gps_synced_accept', 'gps_synced_delivery', 'accept_to_delivery_min', 'accept_hour', 'accept_day', 'accept_month', 'time_of_day', 'gps_drift', 'delivery_speed_proxy', 'delivery_month']


In [38]:
df.to_csv('delivery_cq_cleaned.csv', index=False)
print('File saved as: delivery_Jl_cleaned.csv')
print(f'Total rows saved: {len(df):,}')

File saved as: delivery_Jl_cleaned.csv
Total rows saved: 30,460
